In [ ]:
import streamlit as st
import pandas as pd

# 1. Page Configuration
st.set_page_config(page_title="Tuition Model Prototyper", layout="wide")

st.title("Net Tuition Revenue: Block vs. Per-Credit Model")
st.markdown("""
This tool simulates the financial impact of moving from a flat block rate to a per-credit hour model. 
**Instructions:** Upload your enrollment spreadsheet in the sidebar to begin.
""")

# 2. Sidebar: File Upload
st.sidebar.header("1. Data Input")
uploaded_file = st.sidebar.file_uploader("Upload Enrollment Excel File", type=["xlsx"])

# 3. App Logic (Only runs if a file is uploaded)
if uploaded_file is not None:
    try:
        # Load the data
        df = pd.read_excel(uploaded_file)
        
        # Clean column names
        df.columns = df.columns.str.strip()
        
        # Check for required columns
        required_cols = ['StudentID', 'Discount Rate', 'Block Rate', 'Credits Taken']
        if not all(col in df.columns for col in required_cols):
            st.error(f"Spreadsheet is missing one or more required columns: {required_cols}")
            st.stop()

        # Sidebar: Model Levers
        st.sidebar.header("2. Model Levers")
        
        # Proposed rate input
        new_rate = st.sidebar.number_input(
            "Proposed Per-Credit Rate ($)", 
            min_value=0, 
            value=2200, 
            step=50
        )
        
        # Discount rate slider
        current_avg_disc = float(df['Discount Rate'].mean())
        new_discount = st.sidebar.slider(
            "Global Discount Rate (%)", 
            0.0, 1.0, current_avg_disc,
            help="Adjusting this changes the scholarship percentage applied to the new model."
        )

        # 4. Calculations
        # Current State
        df['Current_NTR'] = df['Block Rate'] * (1 - df['Discount Rate'])
        
        # Proposed State
        df['Proposed_Gross'] = df['Credits Taken'] * new_rate
        df['Proposed_NTR'] = df['Proposed_Gross'] * (1 - new_discount)
        
        # Revenue Difference
        df['Revenue_Delta'] = df['Proposed_NTR'] - df['Current_NTR']

        # 5. Dashboard Metrics
        total_current = df['Current_NTR'].sum()
        total_proposed = df['Proposed_NTR'].sum()
        total_delta = total_proposed - total_current
        pct_change = (total_delta / total_current) * 100 if total_current != 0 else 0

        # KPI Cards
        c1, c2, c3 = st.columns(3)
        c1.metric("Current Net Revenue", f"${total_current:,.0f}")
        c2.metric("Proposed Net Revenue", f"${total_proposed:,.0f}", f"${total_delta:,.0f}")
        c3.metric("Percent Change", f"{pct_change:.2f}%")

        st.divider()

        # 6. Visualizations
        col_left, col_right = st.columns(2)

        with col_left:
            st.subheader("Average Revenue Change by Credit Load")
            # Grouping to see the impact per credit hour level
            impact_by_credit = df.groupby('Credits Taken')['Revenue_Delta'].mean()
            st.bar_chart(impact_by_credit)

        with col_right:
            st.subheader("Student Population Distribution")
            # Show how many students are in each credit bucket
            credit_counts = df['Credits Taken'].value_counts().sort_index()
            st.bar_chart(credit_counts)

        # 7. Data Preview
        with st.expander("View Detailed Student-Level Calculations"):
            st.dataframe(
                df[['StudentID', 'Credits Taken', 'Current_NTR', 'Proposed_NTR', 'Revenue_Delta']],
                use_container_width=True
            )

    except Exception as e:
        # This catches any errors within the 'try' block above
        st.error(f"Error processing data: {e}")

else:
    # This shows before the user uploads their file
    st.info("Waiting for file upload... Please upload the '.xlsx' file in the sidebar.")
    
    st.write("### Required Spreadsheet Format")
    st.write("Your Excel file must contain at least these columns:")
    example_df = pd.DataFrame({
        'StudentID': [1001, 1002],
        'Credits Taken': [12, 18],
        'Block Rate': [35000, 35000],
        'Discount Rate': [0.45, 0.20]
    })
    st.table(example_df)